In [6]:
#!pip install sympy

In [7]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from Sympy_OPF_LALM_class import extract_qhd_solution_vector
from Sympy_OPF_LALM_class import initialize_qhd_acopf_log
from Sympy_OPF_LALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json

In [8]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # Convert "k1", "k2", ... into integer keys 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens


In [9]:
# Initialize the model.
# model = SympyACOPFModel()   # Enable the proximal option if needed later.
n = 3 # Select the test system size: 2, 3, or larger.

if n == 2:
    # 2-bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3-bus model
    model = SympyACOPFModel()

else:
    # n-bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 3 buses 3 lines and 2 generators.


In [ ]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 5.0
alpha = 5e-3   # 对偶步长尽量小一点
max_outer = 100
tol = 1e-4
option = 1  # 1: QHD, 2: Gurobi
qhd_solver = "simbi"  # openjij / simbi

# ========= 新增：初始化日志文件 =========
log_file = initialize_qhd_acopf_log(model, folder="logs", option=option, qhd_solver=qhd_solver)
print("Log file:", log_file)

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=10.0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=20,
                agents=4000,
                max_steps=5000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                penalty_coefficient=0.1,
                ballistic=True,
                heated=True,
                best_only=False,
                verbose=True
            )
        else:
            qhd_model.openjij_setup(
                resolution=6,
                shots=2048,
                sampler_name="SQASampler",
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 10000,
                    "reinitialize_state": True,
                },
            )
        response = qhd_model.optimize(refine=True, verbose=0)
        #response = qhd_model.optimize(refine=False, verbose=0)

        x_new = extract_qhd_solution_vector(response, expected_len=len(variable_list))

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1024
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    log_file = PrintQHDACOPFResults(
        model,
        x_new,
        log_file=log_file,
        iteration=k,
        folder="logs",
        print_to_console=True,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-3_04-13-2026_14-47-10.txt

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---


Bifurcated agents:  73%|███████▎  | 2922/4000 [00:03<00:01, 846.95it/s]


backend time consumption: 3.459993839263916
Minimizer: [0.   0.   0.25 0.25 0.5  0.5  0.5  0.5  0.5  0.5  0.   0.   0.   0.5
 0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.   0.   0.
 0.   0.   0.  ]
||h(x)|| = 0.8827380585016057
[rho-check] ||h_old||=5.181e-01, ||h_new||=8.827e-01, rho=5
Increasing rho to 10.0
Constraint check: False
Update time: 2026-04-13 14:48:10
Iteration: 0
rho: 10
alpha: 0.005
objective_value: 0
feasible: False
max_abs_h: 4.562170425949e-01
l2_norm_h: 8.827380585016e-01
lambda_inf_norm: 2.281085212975e+00
lambda_l2_norm: 4.413690292508e+00
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	0.635136	0.016081	0.925397	0.099576	0.045922	0.000000	0.000000
2	0.635490	0.016861	0.900000	0.103758	-0.009222	0.000000	0.000000
3	0.630591	0.003020	0.924051	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		Lo

Bifurcated agents:  60%|██████    | 2417/4000 [00:03<00:02, 720.68it/s]


backend time consumption: 3.362140655517578
Minimizer: [0.   0.   0.   0.   0.5  0.5  0.25 0.5  0.5  0.5  0.   0.   0.   0.5
 0.5  0.5  0.5  0.5  0.5  0.5  0.5  0.25 0.75 0.5  0.75 0.   0.   0.
 0.   0.   0.  ]
||h(x)|| = 0.4653265984071623
[rho-check] ||h_old||=8.827e-01, ||h_new||=4.653e-01, rho=10
Increasing rho to 20.0
Constraint check: False
Update time: 2026-04-13 14:50:14
Iteration: 1
rho: 20
alpha: 0.005
objective_value: 0
feasible: False
max_abs_h: 2.543894833656e-01
l2_norm_h: 4.653265984072e-01
lambda_inf_norm: 1.507929022018e+00
lambda_l2_norm: 2.811055298329e+00
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.031639	0.004842	0.900000	0.075193	0.002279	0.000000	0.000000
2	1.031684	0.004195	0.900000	0.089236	0.003558	0.000000	0.000000
3	1.012692	-0.018967	0.900000	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		L

...

Bifurcated agents:  99%|█████████▉| 3975/4000 [00:04<00:00, 941.98it/s]


backend time consumption: 4.232120513916016
Minimizer: [0.   0.   0.   0.   0.5  0.5  0.   0.5  0.5  0.5  0.   0.   0.   0.75
 0.25 0.75 0.5  0.75 0.5  0.75 0.25 0.   1.   0.   1.   0.   0.   0.
 0.   0.   0.  ]
||h(x)|| = 2.7305274454280677
[rho-check] ||h_old||=9.632e-01, ||h_new||=2.731e+00, rho=320
Increasing rho to 640.0
Constraint check: False
Update time: 2026-04-13 14:56:32
Iteration: 6
rho: 640
alpha: 0.005
objective_value: 0
feasible: False
max_abs_h: 1.328803792832e+00
l2_norm_h: 2.730527445428e+00
lambda_inf_norm: 4.318816693603e+02
lambda_l2_norm: 8.653938709670e+02
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.064542	-0.062609	0.900000	0.918560	1.569090	0.000000	0.000000
2	0.930998	-0.077870	0.900000	0.000000	-1.936610	0.000000	0.000000
3	0.976368	-0.160626	0.900000	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik	

Bifurcated agents:  54%|█████▍    | 2174/4000 [00:03<00:03, 579.10it/s]


backend time consumption: 3.763125419616699
Minimizer: [0.   1.   0.   1.   1.   0.   0.25 0.5  0.5  0.5  0.   0.5  0.   0.
 1.   0.75 0.25 1.   0.   0.   1.   0.   1.   0.   1.   1.   1.   0.
 0.   0.   0.  ]


KeyboardInterrupt: 